In [1]:
# load in libraries
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
!pip install xlsxwriter
import xlsxwriter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 14.7 MB/s eta 0:00:00


In [26]:
# load in two csvs as df
GOGET_pre_texas_removal = pd.read_csv('/content/drive/MyDrive/GOGET Q4 2024/GOGET oil & gas extraction sites (2024 12 19) - Reserves and Production.csv')
GOGET_post_texas_removal = pd.read_csv('/content/drive/MyDrive/GOGET Q4 2024/GOGET oil & gas extraction sites (main data set) - Reserves and Production.csv')



<ipython-input-26-d5980737d402>:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  GOGET_pre_texas_removal = pd.read_csv('/content/drive/MyDrive/GOGET Q4 2024/GOGET oil & gas extraction sites (2024 12 19) - Reserves and Production.csv')
<ipython-input-26-d5980737d402>:3: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  GOGET_post_texas_removal = pd.read_csv('/content/drive/MyDrive/GOGET Q4 2024/GOGET oil & gas extraction sites (main data set) - Reserves and Production.csv')


In [28]:
print(len(GOGET_pre_texas_removal), len(GOGET_post_texas_removal))

69110 61289


In [29]:
# Example DataFrames
# Replace these with your actual DataFrames
df_original = GOGET_pre_texas_removal

df_modified = GOGET_post_texas_removal

# Identify deleted rows
deleted_rows = pd.concat([df_original, df_modified]).drop_duplicates(keep=False)

# check length of deleted_rows
len(deleted_rows)

57338

In [33]:
df_original = df_original.drop_duplicates(keep='first')  # Or 'last' depending on your need

merged_df = pd.merge(df_original, df_modified, on='unique_id', how='outer', indicator=True)
modified_rows = merged_df[merged_df['_merge'] == 'both'][(df_original != df_modified).any(axis=1)]

KeyError: 'unique_id'

In [ ]:
# Check if all deleted rows contain "Texas"
all_texas_deleted = deleted_rows['State'].str.contains("Texas").all()

# Display the deleted rows and verification
print("Deleted Rows:")
print(deleted_rows)
print(f"All deleted rows contain 'Texas': {all_texas_deleted}")

In [16]:
# Create Unique ID for each row by combining Unit name,Country/Area,"Subnational unit (province, state)","Production/reserves","Fuel description",Reserves classification (original),Quantity (original),Units (original),Data year,Data source

GOGET_post_texas_removal['id'] = GOGET_post_texas_removal['Unit name'] + GOGET_post_texas_removal['Country/Area'] + GOGET_post_texas_removal['Quantity (original)'] + GOGET_post_texas_removal['Units (original)'] + GOGET_post_texas_removal['Data year'] + GOGET_post_texas_removal['Data source']
GOGET_pre_texas_removal['id'] = GOGET_pre_texas_removal['Unit name'] + GOGET_pre_texas_removal['Country/Area'] + GOGET_pre_texas_removal['Quantity (original)'] + GOGET_pre_texas_removal['Units (original)'] + GOGET_pre_texas_removal['Data year'] + GOGET_pre_texas_removal['Data source']

GOGET_pre_texas_removal

,Unit ID,Unit name,Country/Area,"Subnational unit (province, state)",Production/reserves,Fuel description,Reserves classification (original),Quantity (original),Units (original),Data year,Data source,Notes,Reserves classification (converted),Quantity (converted),Units (converted),Data source type,id
0,OG0000001,Matzen,Austria,NaN,cumulative production,oil,NaN,553,million bbl,2014,http://www.searchanddiscovery.com/documents/20...,NaN,NaN,553.0,million bbl,Academic,MatzenAustria553million bbl2014http://www.sear...
1,OG0000001,Matzen,Austria,NaN,reserves,gas,initial hydrocarbons in place,289,million boe,2014,http://www.searchanddiscovery.com/documents/20...,NaN,NaN,49100.0,million m³,Academic,MatzenAustria289million boe2014http://www.sear...
2,OG0000002,Abalone,Brazil,Espírito Santo,cumulative production,oil,NaN,0.3,million bbl,2015,http://www.anp.gov.br/images/planos_desenvolvi...,NaN,NaN,0.3,million bbl,Government,AbaloneBrazil0.3million bbl2015http://www.anp....
3,OG0000002,Abalone,Brazil,Espírito Santo,reserves,gas,original volume in place,3137,million m³,2015,http://www.anp.gov.br/images/planos_desenvolvi...,NaN,NaN,3140.0,million m³,Government,AbaloneBrazil3137million m³2015http://www.anp....
4,OG0000002,Abalone,Brazil,Espírito Santo,reserves,oil,volume in place,33.56,million bbl,2015,http://www.anp.gov.br/images/planos_desenvolvi...,NaN,NaN,33.6,million bbl,Government,AbaloneBrazil33.56million bbl2015http://www.an...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,error,error,Manually classify,NaN
69106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,error,error,Manually classify,NaN
69107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,error,error,Manually classify,NaN
69108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,error,error,Manually classify,NaN


In [20]:
# ignore texas
GOGET_pre_texas_no_texas = GOGET_pre_texas_removal[GOGET_pre_texas_removal['Subnational unit (province, state)'] != 'Texas']

# drop blank unit ID
GOGET_pre_texas_no_texas = GOGET_pre_texas_no_texas.dropna(subset=['id'])
GOGET_post_texas_removal = GOGET_post_texas_removal.dropna(subset=['id'])

In [21]:
# find difference between GOGET_pre_texas_no_texas and GOGET_post_texas_removal
difference = pd.concat([GOGET_pre_texas_no_texas, GOGET_post_texas_removal]).drop_duplicates(keep=False)

In [23]:
difference

,Unit ID,Unit name,Country/Area,"Subnational unit (province, state)",Production/reserves,Fuel description,Reserves classification (original),Quantity (original),Units (original),Data year,Data source,Notes,Reserves classification (converted),Quantity (converted),Units (converted),Data source type,id
21409,OG0013575,Maidstone Sparky,Canada,Saskatchewan,production,oil,NaN,400,m³/y,2017,Saskatchewan Reserves,NaN,NaN,0.00252,million bbl/y,Government,Maidstone SparkyCanada400m³/y2017Saskatchewan ...
29745,OG0017456,Banga Kayo (Phase 1),Republic of the Congo,NaN,production design capacity,hydrocarbons,NaN,20000.00,bbl/d,2018,https://iea.blob.core.windows.net/assets/73f2c...,NaN,NaN,7.3,million boe/y,Intergovernmental Organization,Banga Kayo (Phase 1)Republic of the Congo20000...
29746,OG0017456,Banga Kayo (Phase 1),Republic of the Congo,NaN,reserves,hydrocarbons,remaining recoverable reserves,195.6,million boe,2021,https://www.offshore-technology.com/marketdata...,NaN,NaN,196.0,million boe,Commercial Data Provider,Banga Kayo (Phase 1)Republic of the Congo195.6...
36189,OG0002571,Emú,Mexico,Tamaulipas,production,nonassociated gas,NaN,2.731,million cubic feet/d,2022,SIH2023,NaN,NaN,28.2,million m³/y,Government,EmúMexico2.731million cubic feet/d2022SIH2023
36190,OG0002605,Enlace,Mexico,Tamaulipas,production,nonassociated gas,NaN,1.457,million cubic feet/d,2022,SIH2023,NaN,NaN,15.1,million m³/y,Government,EnlaceMexico1.457million cubic feet/d2022SIH2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57713,OG0013427,Froude Red River,Canada,Saskatchewan,production,oil,NaN,456.60,m³/y,2023,SaskOil2023,NaN,NaN,2.87E-03,million bbl/y,Government,Froude Red RiverCanada456.60m³/y2023SaskOil2023
57714,OG0013530,Lashburn Mclaren,Canada,Saskatchewan,production,oil,NaN,454.50,m³/y,2023,SaskOil2023,NaN,NaN,2.86E-03,million bbl/y,Government,Lashburn MclarenCanada454.50m³/y2023SaskOil2023
57715,OG0013376,Elswick South Frobisher,Canada,Saskatchewan,production,oil,NaN,429.8,m³/y,2023,SaskOil2023,NaN,NaN,2.70E-03,million bbl/y,Government,Elswick South FrobisherCanada429.8m³/y2023Sask...
58827,OG0010406,Moon,Australia,Queensland,reserves,gas,2p reserves,1.089513513,petajoules,2022,Queensland_2020-2024_Reserves,total conventional,2P reserves,3.05E+01,million m³,Government,MoonAustralia1.089513513petajoules2022Queensla...


In [4]:


# Identify rows in GOGET_pre_texas_no_texas that are missing from GOGET_post_texas_no_texas
# Assuming a unique identifier for each row (e.g., 'id' or a combination of columns)
# Replace 'id' with the actual unique identifier column name or combination of columns if needed.
if 'id' in GOGET_pre_texas_no_texas.columns:
  missing_rows = GOGET_pre_texas_no_texas[~GOGET_pre_texas_no_texas['id'].isin(GOGET_post_texas_no_texas['id'])]
elif all(col in GOGET_pre_texas_no_texas.columns for col in ['Production/reserves','Fuel description','Quantity (original)', 'Units (original)', 'Data source', 'Quantity (converted)']):
    missing_rows = pd.merge(GOGET_pre_texas_no_texas, GOGET_post_texas_no_texas, on=['Production/reserves','Fuel description','Quantity (original)', 'Units (original)', 'Data source', 'Quantity (converted)'], how='left', indicator=True)
    missing_rows = missing_rows[missing_rows['_merge'] == 'left_only']

else:
  print("No unique identifier found, and site, lat and long not present. Counting based on all columns may not be accurate.")
  missing_rows = pd.concat([GOGET_pre_texas_no_texas, GOGET_post_texas_no_texas]).drop_duplicates(keep=False)


# Count the number of missing rows
count_missing_rows = len(missing_rows)

# Print the count
print(f"Number of rows in GOGET_pre_texas (excluding Texas) missing from GOGET_post_texas (excluding Texas): {count_missing_rows}")

Number of rows in GOGET_pre_texas (excluding Texas) missing from GOGET_post_texas (excluding Texas): 24843
